# Temperature Missingness Analysis

## Importing Necessities

In [17]:
import pandas as pd
from pathlib import Path

## File paths

In [19]:
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'

baseline_file = DATA_RAW / 'weather_baseline_1981_2010_jul_aug.csv'
analysis_file = DATA_RAW / 'weather_analysis_2015_2025_all_months.csv'

## Loading the weather datasets

In [20]:
baseline = pd.read_csv(baseline_file)
analysis = pd.read_csv(analysis_file)

print('Baseline shape:', baseline.shape)
print('Analysis shape:', analysis.shape)

baseline.head()

Baseline shape: (84379, 7)
Analysis shape: (166427, 7)


,parish,station_id,station_name,date,TAVG,TMAX,TMIN
0,Acadia,GHCND:USC00162212,"CROWLEY 2 NE, LA US",1981-07-01,NaN,91.0,73.0
1,Acadia,GHCND:USC00162212,"CROWLEY 2 NE, LA US",1981-07-02,NaN,93.0,75.0
2,Acadia,GHCND:USC00162212,"CROWLEY 2 NE, LA US",1981-07-03,NaN,81.0,69.0
3,Acadia,GHCND:USC00162212,"CROWLEY 2 NE, LA US",1981-07-04,NaN,88.0,72.0
4,Acadia,GHCND:USC00162212,"CROWLEY 2 NE, LA US",1981-07-05,NaN,86.0,74.0


## Inspecting the temperature columns

In [21]:
baseline[['TMIN', 'TMAX', 'TAVG']].head()

,TMIN,TMAX,TAVG
0,73.0,91.0,NaN
1,75.0,93.0,NaN
2,69.0,81.0,NaN
3,72.0,88.0,NaN
4,74.0,86.0,NaN


## Missingness summary before imputation

In [23]:
def missing_summary(df, name):
    summary = pd.DataFrame({
        'missing_count': df[['TMIN', 'TMAX', 'TAVG']].isna().sum(),
        'missing_percent': df[['TMIN', 'TMAX', 'TAVG']].isna().mean() * 100
    })
    print(f'\n{name} missingness before imputation:')
    print(summary)
    return summary

baseline_missing_before = missing_summary(baseline, 'Baseline')
analysis_missing_before = missing_summary(analysis, 'Analysis')


Baseline missingness before imputation:
      missing_count  missing_percent
TMIN            219         0.259543
TMAX             24         0.028443
TAVG          79309        93.991396

Analysis missingness before imputation:
      missing_count  missing_percent
TMIN            297         0.178457
TMAX            220         0.132190
TAVG         156805        94.218486


## Counting how many TAVG values can be filled

A `TAVG` value is considered fillable if:
- `TAVG` is missing
- `TMIN` is present
- `TMAX` is present

In [24]:
def count_fillable_tavg(df, name):
    fillable = df['TAVG'].isna() & df['TMIN'].notna() & df['TMAX'].notna()
    print(f'{name} fillable TAVG rows:', fillable.sum())
    return fillable

baseline_fillable_mask = count_fillable_tavg(baseline, 'Baseline')
analysis_fillable_mask = count_fillable_tavg(analysis, 'Analysis')

Baseline fillable TAVG rows: 79066
Analysis fillable TAVG rows: 156288


## Imputing TAVG using (TMIN + TMAX) / 2 where possible

In [25]:
def fill_tavg(df):
    df = df.copy()
    mask = df['TAVG'].isna() & df['TMIN'].notna() & df['TMAX'].notna()
    df.loc[mask, 'TAVG'] = (df.loc[mask, 'TMIN'] + df.loc[mask, 'TMAX']) / 2
    return df

baseline_clean = fill_tavg(baseline)
analysis_clean = fill_tavg(analysis)


## Missingness summary after imputation

In [28]:
baseline_missing_after = missing_summary(baseline_clean, 'Baseline')
analysis_missing_after = missing_summary(analysis_clean, 'Analysis')

baseline_missing_after


Baseline missingness before imputation:
      missing_count  missing_percent
TMIN            219         0.259543
TMAX             24         0.028443
TAVG            243         0.287986

Analysis missingness before imputation:
      missing_count  missing_percent
TMIN            297         0.178457
TMAX            220         0.132190
TAVG            517         0.310647


,missing_count,missing_percent
TMIN,219,0.259543
TMAX,24,0.028443
TAVG,243,0.287986


## Comparing missingness before vs after

In [27]:
baseline_comparison = baseline_missing_before.join(
    baseline_missing_after,
    lsuffix='_before',
    rsuffix='_after'
)

analysis_comparison = analysis_missing_before.join(
    analysis_missing_after,
    lsuffix='_before',
    rsuffix='_after'
)

print('Baseline comparison:')
display(baseline_comparison)

print('Analysis comparison:')
display(analysis_comparison)


Baseline comparison:


,missing_count_before,missing_percent_before,missing_count_after,missing_percent_after
TMIN,219,0.259543,219,0.259543
TMAX,24,0.028443,24,0.028443
TAVG,79309,93.991396,243,0.287986


Analysis comparison:


,missing_count_before,missing_percent_before,missing_count_after,missing_percent_after
TMIN,297,0.178457,297,0.178457
TMAX,220,0.132190,220,0.132190
TAVG,156805,94.218486,517,0.310647


## 10. Save cleaned datasets back to the same files

This overwrites the current weather files with the cleaned versions where only `TAVG` has been imputed when justified.

In [ ]:
baseline_clean.to_csv(baseline_file, index=False)
analysis_clean.to_csv(analysis_file, index=False)

print('Updated baseline file saved to:', baseline_file)
print('Updated analysis file saved to:', analysis_file)


## 11. Report-ready interpretation

Average daily temperature (`TAVG`) values missing from the NOAA weather datasets were imputed only when both minimum (`TMIN`) and maximum (`TMAX`) daily temperature values were available. In those cases, `TAVG` was calculated as the arithmetic mean of `TMIN` and `TMAX`. Missing `TMIN` and `TMAX` values were not imputed at this stage because they could not be inferred as reliably without introducing stronger assumptions.